In [19]:
from neo4j import GraphDatabase
from dotenv import load_dotenv

import pandas as pd
import os

from pathlib import Path

In [20]:


uri = "bolt://localhost:7687"

username = "neo4j"

password = os.getenv("NEO4J_PASSWORD")

driver = GraphDatabase.driver(
    uri,
    auth=(username, password)
)

driver.verify_connectivity()

print("Connected to Neo4j successfully.")

Connected to Neo4j successfully.


In [21]:
# ============================================================
# Reusable Neo4j query function
# ============================================================

def run_query(query):

    with driver.session() as session:

        result = session.run(query)

        return [
            record.data()
            for record in result
        ]

In [22]:
# ============================================================
# Verify graph nodes
# ============================================================

node_summary = pd.DataFrame(

    run_query("""

    MATCH (n)

    RETURN
        labels(n)[0] AS node_type,
        COUNT(n) AS total

    ORDER BY total DESC

    """)

)

print("Graph nodes verified successfully.")

node_summary

Graph nodes verified successfully.


,node_type,total
0,Order,65752
1,Customer,20652
2,Product,118
3,Region,23
4,Department,11
5,ShippingMode,4


In [23]:
# ============================================================
# Verify graph relationships
# ============================================================

relationship_summary = pd.DataFrame(

    run_query("""

    MATCH ()-[r]->()

    RETURN
        type(r) AS relationship,
        COUNT(r) AS total

    ORDER BY total DESC

    """)

)

print("Graph relationships verified successfully.")

relationship_summary

Graph relationships verified successfully.


,relationship,total
0,CONTAINS,159763
1,PLACED,65752
2,SHIPPED_TO,65752
3,USED_MODE,65752
4,IN_DEPARTMENT,118


In [24]:
# ============================================================
# Drop existing graph projection if present
# ============================================================

run_query("""

CALL gds.graph.drop(
    'supply_chain_graph',
    false
)

""")

print("Existing graph projection dropped.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=3, column=1, offset=2>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 2, 'line': 3, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n\nCALL gds.graph.drop(\n    'supply_chain_graph',\n    false\n)\n\n"


Existing graph projection dropped.


In [25]:
run_query("""

CALL gds.graph.project(

    'supply_chain_graph',

    [
        'Customer',
        'Department',
        'Order',
        'Product',
        'Region',
        'ShippingMode'
    ],

    {
        PLACED: {
            orientation: 'UNDIRECTED'
        },

        CONTAINS: {
            orientation: 'UNDIRECTED'
        },

        IN_DEPARTMENT: {
            orientation: 'UNDIRECTED'
        },

        SHIPPED_TO: {
            orientation: 'UNDIRECTED'
        },
          
        USED_MODE: {
            orientation: 'UNDIRECTED'
        }
    }

)

""")

print("Graph projection created successfully.")

Graph projection created successfully.


In [26]:
projection_df = pd.DataFrame(

    run_query("""

    CALL gds.graph.list()

    YIELD
        graphName,
        nodeCount,
        relationshipCount

    RETURN
        graphName,
        nodeCount,
        relationshipCount

    """)

)

projection_df

,graphName,nodeCount,relationshipCount
0,supply_chain_graph,86560,714274


In [27]:
run_query("""

CALL gds.pageRank.write(

    'supply_chain_graph',

    {
        writeProperty: 'pagerank'
    }

)

""")

print("PageRank completed")

PageRank completed


In [40]:
# pagerank_df = pd.DataFrame(

#     run_query("""

#     MATCH (o:Order)

#     WHERE o.pagerank IS NOT NULL

#     RETURN

#         o.id AS order_id,

#         o.order_status AS order_status,

#         o.shipping_mode AS shipping_mode,

#         o.delay AS delay,

#         o.pagerank AS pagerank

#     ORDER BY pagerank DESC

#     LIMIT 20

#     """)

# )

# pagerank_df

pagerank_df = pd.DataFrame(

    run_query("""

    MATCH (o:Order)

    WHERE o.pagerank IS NOT NULL

    RETURN

        o.id AS order_id,

        o.pagerank AS pagerank

    """)

)

pagerank_df.head()

,order_id,pagerank
0,77202,0.731489
1,75939,0.727068
2,75938,0.727068
3,75937,0.726586
4,75936,0.726586


In [41]:
degree_df = pd.DataFrame(

    run_query("""

    MATCH (o:Order)

    OPTIONAL MATCH (o)-[r]-()

    RETURN

        o.id AS order_id,

        COUNT(r) AS degree,

        COUNT {
            (o)-[]->()
        } AS out_degree,

        COUNT {
            (o)<-[]-()
        } AS in_degree

    """)
)

degree_df.head()

,order_id,degree,out_degree,in_degree
0,77202,4,3,1
1,75939,4,3,1
2,75938,4,3,1
3,75937,4,3,1
4,75936,4,3,1


In [42]:
pagerank_df.head()

,order_id,pagerank
0,77202,0.731489
1,75939,0.727068
2,75938,0.727068
3,75937,0.726586
4,75936,0.726586


In [43]:
run_query("""

CALL gds.louvain.write(

    'supply_chain_graph',

    {
        writeProperty: 'community'
    }

)

""")

print("Community detection completed.")

Community detection completed.


In [44]:
# community_df = pd.DataFrame(

#     run_query("""

#     MATCH (o:Order)

#     WHERE o.community IS NOT NULL

#     RETURN

#         o.id AS order_id,

#         o.shipping_mode AS shipping_mode,

#         o.delay AS delay,

#         o.community AS community

#     LIMIT 20

#     """)

# )

# community_df

community_df = pd.DataFrame(

    run_query("""

    MATCH (o:Order)

    WHERE o.community IS NOT NULL

    RETURN

        o.id AS order_id,

        o.community AS community

    """)

)

community_df.head()

,order_id,community
0,77202,13067
1,75939,13067
2,75938,13067
3,75937,13067
4,75936,13067


In [46]:
degree_df["order_id"] = degree_df["order_id"].astype(str)

pagerank_df["order_id"] = pagerank_df["order_id"].astype(str)

community_df["order_id"] = community_df["order_id"].astype(str)

In [47]:
graph_features_df = degree_df.merge(

    pagerank_df[
        ["order_id", "pagerank"]
    ],

    on="order_id",

    how="left"
)

graph_features_df = graph_features_df.merge(

    community_df[
        ["order_id", "community"]
    ],

    on="order_id",

    how="left"
)

graph_features_df.head()

,order_id,degree,out_degree,in_degree,pagerank,community
0,77202,4,3,1,0.731489,13067
1,75939,4,3,1,0.727068,13067
2,75938,4,3,1,0.727068,13067
3,75937,4,3,1,0.726586,13067
4,75936,4,3,1,0.726586,13067


In [48]:
graph_features_df.to_csv(

    "../data/clean/graph_features.csv",

    index=False
)

print("graph_features.csv saved successfully.")

graph_features.csv saved successfully.


In [49]:
run_query("""

CALL gds.graph.drop(
    'supply_chain_graph'
)

""")

print("Graph projection dropped.")

Received notification from DBMS server: <GqlStatusObject gql_status='01N03', status_description='warn: procedure field deprecated. The field `schema` of procedure gds.graph.drop() is deprecated.', position=<SummaryInputPosition line=3, column=1, offset=2>, raw_classification='DEPRECATION', classification=<NotificationClassification.DEPRECATION: 'DEPRECATION'>, raw_severity='WARNING', severity=<NotificationSeverity.WARNING: 'WARNING'>, diagnostic_record={'_classification': 'DEPRECATION', '_severity': 'WARNING', '_position': {'offset': 2, 'line': 3, 'column': 1}, 'OPERATION': '', 'OPERATION_CODE': '0', 'CURRENT_SCHEMA': '/'}> for query: "\n\nCALL gds.graph.drop(\n    'supply_chain_graph'\n)\n\n"


Graph projection dropped.


In [50]:
driver.close()

print("Notebook 3 completed successfully.")

Notebook 3 completed successfully.
